In [7]:
!pip install --upgrade pip
!pip install opencv-python-headless numpy pandas scikit-learn matplotlib

import os
import cv2
import numpy as np
import pandas as pd
import torch
import torchvision
from pathlib import Path
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

PyTorch Version: 2.14.1+cpu
GPU Available: False


In [8]:
import shutil

def prepare_visa_dataset(split_file, data_folder, save_folder):
    df = pd.read_csv(split_file)
    os.makedirs(save_folder, exist_ok=True)

    for idx, row in df.iterrows():
        image_path = os.path.join(data_folder, row.get('image', ''))
        split_type = row.get('split', 'train')
        label = row.get('label', 'good')
        object_class = row.get('object', 'candle')

        if split_type == 'train':
            dest_dir = os.path.join(save_folder, object_class, 'train', 'good')
        elif label == 'good':
            dest_dir = os.path.join(save_folder, object_class, 'test', 'good')
        else:
            dest_dir = os.path.join(save_folder, object_class, 'test', 'bad')

        os.makedirs(dest_dir, exist_ok=True)
        if os.path.exists(image_path):
            shutil.copy(image_path, dest_dir)

    print("Dataset organized successfully at:", save_folder)

# Usage Example:
# prepare_visa_dataset('/content/drive/MyDrive/VisA/split_csv/1cls.csv', '/content/drive/MyDrive/VisA', '/content/drive/MyDrive/VisA_pytorch/1cls')

In [9]:
def preprocess_image(image_path, target_size=(256, 256)):
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Image not found: {image_path}")

    # 1. Resize
    img_resized = cv2.resize(img, target_size)

    # 2. CLAHE (Contrast Limited Adaptive Histogram Equalization)
    lab = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    img_clahe = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # 3. Gaussian Filtering for noise reduction
    img_filtered = cv2.GaussianBlur(img_clahe, (3, 3), 0)

    # 4. Normalization (scaling to [0, 1])
    img_normalized = img_filtered.astype(np.float32) / 255.0

    return img_normalized

In [10]:
def generate_synthetic_defect(image):
    h, w, _ = image.shape
    defect_img = image.copy()
    mask = np.zeros((h, w), dtype=np.uint8)

    # CutPaste style simulation: copy a random patch and paste it elsewhere
    patch_size = 32
    y1 = np.random.randint(0, h - patch_size)
    x1 = np.random.randint(0, w - patch_size)
    y2 = np.random.randint(0, h - patch_size)
    x2 = np.random.randint(0, w - patch_size)

    patch = defect_img[y1:y1+patch_size, x1:x1+patch_size]
    defect_img[y2:y2+patch_size, x2:x2+patch_size] = patch

    # Update mask
    mask[y2:y2+patch_size, x2:x2+patch_size] = 255

    return defect_img, mask

In [12]:
class VisADataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths = list(self.root_dir.glob('*/*/*.png')) + list(self.root_dir.glob('*/*/*.jpg'))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = str(self.image_paths[idx])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (256, 256))

        if self.transform:
            image = self.transform(image)
        else:
            image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0

        return image, img_path

# DataLoader Usage Example:
# dataset = VisADataset(root_dir='/content/drive/MyDrive/VisA_pytorch/1cls/candle/train')
# dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [16]:
from google.colab import drive
import os
import pandas as pd
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Define exact paths based on your current Drive structure
split_file_path = '/content/drive/MyDrive/Visa/1cls.csv'
data_folder_path = '/content/drive/MyDrive/Visa/VisA_20220922'
save_folder_path = '/content/drive/MyDrive/Visa/VisA_pytorch'

def organize_visa_dataset(split_file, data_folder, save_folder):
    if not os.path.exists(split_file):
        print(f"Error: Split CSV file not found at: {split_file}")
        return

    df = pd.read_csv(split_file)
    os.makedirs(save_folder, exist_ok=True)

    success_count = 0
    missing_count = 0

    for idx, row in df.iterrows():
        image_rel_path = row.get('image', '')
        image_path = os.path.join(data_folder, image_rel_path)
        split_type = row.get('split', 'train')
        label = row.get('label', 'good')
        object_class = row.get('object', 'candle')

        # Build destination directory structure
        if split_type == 'train':
            dest_dir = os.path.join(save_folder, '1cls', object_class, 'train', 'good')
        elif label == 'good':
            dest_dir = os.path.join(save_folder, '1cls', object_class, 'test', 'good')
        else:
            dest_dir = os.path.join(save_folder, '1cls', object_class, 'test', 'bad')

        os.makedirs(dest_dir, exist_ok=True)

        if os.path.exists(image_path):
            # Use copy2 to preserve metadata, or move if preferred
            shutil.copy2(image_path, dest_dir)
            success_count += 1
        else:
            missing_count += 1

    print(f"Process completed!")
    print(f"Successfully organized: {success_count} images")
    print(f"Missing images: {missing_count}")
    print(f"Target directory: {save_folder}")

# Execute the organization function
organize_visa_dataset(split_file_path, data_folder_path, save_folder_path)

Mounted at /content/drive
Process completed!
Successfully organized: 10821 images
Missing images: 0
Target directory: /content/drive/MyDrive/Visa/VisA_pytorch


In [17]:
import os
import cv2
import numpy as np
from glob import glob

# 1. Define paths for structured data
base_dir = '/content/drive/MyDrive/Visa/VisA_pytorch/1cls'

def apply_clahe_and_normalize(image_path):
    # Read image in grayscale or RGB depending on pipeline needs (here using RGB / BGR via OpenCV)
    img = cv2.imread(image_path)
    if img is None:
        return

    # Convert to LAB color space to apply CLAHE on the L channel (luminance)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # Initialize CLAHE (Contrast Limited Adaptive Histogram Equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    # Merge channels back and convert back to BGR
    limg = cv2.merge((cl, a, b))
    enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # Normalization (scaling pixel values to [0, 1] or standard PyTorch tensors preparation)
    normalized_img = enhanced_img.astype(np.float32) / 255.0

    # Overwrite the image or save to a preprocessed directory (here we overwrite in-place for efficiency)
    # Alternatively, scale back to uint8 if saving as standard image files:
    final_img = (normalized_img * 255).astype(np.uint8)
    cv2.imwrite(image_path, final_img)

def process_dataset(root_dir):
    # Find all images inside the structured dataset folders
    image_paths = glob(os.path.join(root_dir, '**', '*.JPG'), recursive=True)
    # Also check for lowercase extensions if any (.jpg / .png)
    image_paths.extend(glob(os.path.join(root_dir, '**', '*.jpg'), recursive=True))
    image_paths.extend(glob(os.path.join(root_dir, '**', '*.png'), recursive=True))

    count = 0
    for img_path in image_paths:
        apply_clahe_and_normalize(img_path)
        count += 1

    print(f"Preprocessing completed! Successfully processed {count} images.")

# Execute preprocessing
process_dataset(base_dir)

Preprocessing completed! Successfully processed 10774 images.


In [22]:
import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm

def apply_cutpaste(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return

    h, w, _ = img.shape
    ph, pw = int(h * 0.15), int(w * 0.15)
    if ph < 1 or pw < 1:
        return

    y1 = np.random.randint(0, h - ph)
    x1 = np.random.randint(0, w - pw)
    patch = img[y1:y1+ph, x1:x1+pw].copy()

    y2 = np.random.randint(0, h - ph)
    x2 = np.random.randint(0, w - pw)

    augmented_img = img.copy()
    augmented_img[y2:y2+ph, x2:x2+pw] = patch

    base_path = image_path.replace('train/good', 'test/synthetic_bad')
    os.makedirs(os.path.dirname(base_path), exist_ok=True)
    cv2.imwrite(base_path, augmented_img)

def generate_synthetic_defects():
    base_path_root = '/content/drive/MyDrive/Visa/VisA_pytorch/1cls'
    all_good_images = []

    for ext in ['*.JPG', '*.jpg', '*.png']:
        all_good_images.extend(glob(os.path.join(base_path_root, '*', 'train', 'good', ext)))

    print(f"Found {len(all_good_images)} training images for CutPaste generation.")

    if len(all_good_images) == 0:
        print("Warning: No images found! Please check folder path.")
        return

    for img_path in tqdm(all_good_images, desc="Generating CutPaste Defects", unit="img"):
        apply_cutpaste(img_path)

    print("Synthetic defect generation completed successfully!")

generate_synthetic_defects()

Found 8659 training images for CutPaste generation.


Generating CutPaste Defects: 100%|██████████| 8659/8659 [08:39<00:00, 16.67img/s]

Synthetic defect generation completed successfully!


In [24]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
epochs = 5

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# البحث التلقائي عن مسار الـ train الصحيح لتجنب أي خطأ
base_path_root = '/content/drive/MyDrive/Visa/VisA_pytorch/1cls'
train_dir = None

# التحقق من مكان وجود مجلد train
if os.path.exists(os.path.join(base_path_root, 'train')):
    train_dir = os.path.join(base_path_root, 'train')
else:
    # إذا كان موزعا داخل المجلدات الفرعية
    subdirs = [os.path.join(base_path_root, d, 'train') for d in os.listdir(base_path_root) if os.path.isdir(os.path.join(base_path_root, d))]
    for s in subdirs:
        if os.path.exists(s):
            train_dir = os.path.dirname(s) # أو استخدام المجلد الأب حسب الهيكلة
            break

# إذا لم يتم إيجاده تلقائياً، نستخدم المسار المباشر الأول
if train_dir is None:
    train_dir = base_path_root

print(f"Using dataset path: {train_dir}")

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

print(f"Dataset loaded for training. Total training batches: {len(train_loader)}")

class AnomalyAutoencoder(nn.Module):
    def __init__(self):
        super(AnomalyAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

model = AnomalyAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Starting Model Training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]", unit="batch")
    for images, _ in progress_bar:
        images = images.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"Epoch [{epoch+1}/{epochs}] Completed. Average Loss: {running_loss/len(train_loader):.4f}")

print("Training completed successfully! Model is ready.")

Using dataset path: /content/drive/MyDrive/Visa/VisA_pytorch/1cls/candle
Dataset loaded for training. Total training batches: 63
Starting Model Training...


Epoch [1/5]: 100%|██████████| 63/63 [01:46<00:00,  1.69s/batch, loss=1.3204]


Epoch [1/5] Completed. Average Loss: 1.7329


Epoch [2/5]: 100%|██████████| 63/63 [01:55<00:00,  1.84s/batch, loss=1.2231]


Epoch [2/5] Completed. Average Loss: 1.2846


Epoch [3/5]: 100%|██████████| 63/63 [01:58<00:00,  1.89s/batch, loss=1.2185]


Epoch [3/5] Completed. Average Loss: 1.2198


Epoch [4/5]: 100%|██████████| 63/63 [02:05<00:00,  2.00s/batch, loss=1.2185]


Epoch [4/5] Completed. Average Loss: 1.2177


Epoch [5/5]: 100%|██████████| 63/63 [01:52<00:00,  1.79s/batch, loss=1.2252]

Epoch [5/5] Completed. Average Loss: 1.2171
Training completed successfully! Model is ready.


In [29]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

# 1. Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32
epochs = 5

# 2. Data Transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Smart Path Resolution (Works on Colab and Locally without breaking)
if os.path.exists('/content/drive/MyDrive/Visa/VisA_pytorch/1cls'):
    base_path_root = '/content/drive/MyDrive/Visa/VisA_pytorch/1cls/candle'
elif os.path.exists('/content/1cls'):
    base_path_root = '/content/1cls/candle'
else:
    base_path_root = os.path.join(os.getcwd(), '1cls', 'candle')

train_dir = base_path_root
print(f"Using dataset path: {train_dir}")

# 4. Dataset & DataLoader
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

print(f"Dataset loaded successfully. Total training batches: {len(train_loader)}")

# 5. Autoencoder Model Architecture
class AnomalyAutoencoder(nn.Module):
    def __init__(self):
        super(AnomalyAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

model = AnomalyAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 6. Training Loop with Progress Bar
print("Starting Model Training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]", unit="batch")
    for images, _ in progress_bar:
        images = images.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"Epoch [{epoch+1}/{epochs}] Completed. Average Loss: {running_loss/len(train_loader):.4f}")

print("Training completed successfully! Model is ready.")

Using dataset path: /content/drive/MyDrive/Visa/VisA_pytorch/1cls/candle
Dataset loaded successfully. Total training batches: 63
Starting Model Training...


Epoch [1/5]: 100%|██████████| 63/63 [01:39<00:00,  1.58s/batch, loss=1.2692]


Epoch [1/5] Completed. Average Loss: 1.6087


Epoch [2/5]: 100%|██████████| 63/63 [01:42<00:00,  1.63s/batch, loss=1.2017]


Epoch [2/5] Completed. Average Loss: 1.2238


Epoch [3/5]: 100%|██████████| 63/63 [01:53<00:00,  1.79s/batch, loss=1.2195]


Epoch [3/5] Completed. Average Loss: 1.2145


Epoch [4/5]: 100%|██████████| 63/63 [01:45<00:00,  1.67s/batch, loss=1.2208]


Epoch [4/5] Completed. Average Loss: 1.2132


Epoch [5/5]: 100%|██████████| 63/63 [01:43<00:00,  1.65s/batch, loss=1.2057]

Epoch [5/5] Completed. Average Loss: 1.2124
Training completed successfully! Model is ready.
